# Newton's Second Law: $F = ma$

This notebook contains two interactive animations:

1. **A block pushed horizontally across a rough surface**
2. **A block pulled by a rope at an angle**, with the pulling force decomposed into horizontal and vertical components

The animations use the same Matplotlib/Jupyter structure as the working projectile notebook.


In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.lines import Line2D
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display

g = 9.81

def block_dimensions(m):
    # Visual scaling only
    scale = np.sqrt(m / 20.0)
    return 1.5 * scale, 1.2 * scale

def update_arrow(arrow, start, end, visible=True):
    arrow.set_positions(start, end)
    arrow.set_visible(visible)


## 1. Horizontal push with friction

For the sliding block,

$$N = mg$$

$$F_f = \mu N$$

$$F_{net} = F_{push} - F_f$$

$$a = \frac{F_{net}}{m}$$


In [ ]:
# Controls — Animation 1
mass1 = widgets.IntSlider(
    value=20, min=5, max=100, step=5,
    description='Mass (kg):', continuous_update=False,
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

push_force1 = widgets.FloatSlider(
    value=120, min=0, max=500, step=10,
    description='Push force (N):', continuous_update=False,
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

mu1 = widgets.FloatSlider(
    value=0.20, min=0, max=1.0, step=0.05,
    description='Friction μ:', continuous_update=False,
    readout_format='.2f',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

show_forces1 = widgets.Checkbox(value=True, description='Show force arrows')

run_button1 = widgets.Button(description='Run animation', button_style='primary')
reset_button1 = widgets.Button(description='Reset')

display(widgets.VBox([
    mass1,
    push_force1,
    mu1,
    show_forces1,
    widgets.HBox([run_button1, reset_button1])
]))

# Fixed animation frame
X_MIN1, X_MAX1 = 0, 20
Y_MIN1, Y_MAX1 = -3, 7

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.set_xlim(X_MIN1, X_MAX1)
ax1.set_ylim(Y_MIN1, Y_MAX1)
ax1.set_xlabel('Horizontal position (m)')
ax1.set_yticks([])
ax1.set_title('Horizontal Push with Friction')

# Ground
ax1.axhline(0, color='black', linewidth=2)

# Block
w0, h0 = block_dimensions(mass1.value)
block1 = Rectangle(
    (1, 0), w0, h0,
    facecolor='lightsteelblue',
    edgecolor='black',
    linewidth=1.5
)
ax1.add_patch(block1)

# Persistent force arrows
push_arrow1 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                              linewidth=2.2, color='tab:blue')
friction_arrow1 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                  linewidth=2.2, color='tab:red')
normal_arrow1 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                linewidth=2.2, color='tab:green')
weight_arrow1 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                linewidth=2.2, color='tab:purple')

for arrow in [push_arrow1, friction_arrow1, normal_arrow1, weight_arrow1]:
    ax1.add_patch(arrow)

# Fixed values box
info1 = ax1.text(
    0.98, 0.97, '',
    transform=ax1.transAxes,
    ha='right', va='top',
    fontsize=10,
    family='monospace',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9)
)

# Actual legend
legend_handles1 = [
    Line2D([0],[0], color='tab:blue', lw=2.5, label='Applied force'),
    Line2D([0],[0], color='tab:red', lw=2.5, label='Friction'),
    Line2D([0],[0], color='tab:green', lw=2.5, label='Normal reaction'),
    Line2D([0],[0], color='tab:purple', lw=2.5, label='Weight')
]
ax1.legend(handles=legend_handles1, loc='upper left')

animation1 = None

def physics1():
    m = mass1.value
    F = push_force1.value
    mu = mu1.value

    W = m * g
    N = W
    friction_max = mu * N

    if F <= friction_max:
        # Treat friction as balancing the push while stationary
        friction = F
        net = 0
        a = 0
        moving = False
    else:
        friction = friction_max
        net = F - friction
        a = net / m
        moving = True

    return m, F, mu, W, N, friction, net, a, moving

def draw_state1(x=1, t=0, v=0):
    m, F, mu, W, N, friction, net, a, moving = physics1()

    width, height = block_dimensions(m)
    block1.set_x(x)
    block1.set_y(0)
    block1.set_width(width)
    block1.set_height(height)

    cx = x + width / 2
    cy = height / 2
    scale = 0.010
    visible = show_forces1.value

    update_arrow(
        push_arrow1,
        (x + width, cy),
        (x + width + F * scale, cy),
        visible and F > 0
    )

    update_arrow(
        friction_arrow1,
        (x, cy),
        (x - friction * scale, cy),
        visible and friction > 0
    )

    update_arrow(
        normal_arrow1,
        (cx, height),
        (cx, height + N * scale),
        visible and N > 0
    )

    update_arrow(
        weight_arrow1,
        (cx, 0),
        (cx, -W * scale),
        visible and W > 0
    )

    state = 'MOVING' if moving else 'AT REST'

    info1.set_text(
        f'm = {m:.0f} kg\n'
        f'Push = {F:.1f} N\n'
        f'μ = {mu:.2f}\n'
        f'N = {N:.1f} N\n'
        f'Friction = {friction:.1f} N\n'
        f'Net force = {net:.1f} N\n'
        f'a = {a:.2f} m/s²\n'
        f'v = {v:.2f} m/s\n'
        f't = {t:.2f} s\n'
        f'{state}'
    )

    fig1.canvas.draw_idle()

def run_animation1(_=None):
    global animation1

    m, F, mu, W, N, friction, net, a, moving = physics1()

    if not moving or a <= 0:
        draw_state1()
        return

    width, height = block_dimensions(m)
    x0 = 1
    x_end = X_MAX1 - width - 0.5

    t_end = np.sqrt(2 * (x_end - x0) / a)
    t = np.linspace(0, t_end, max(80, int(t_end * 35)))
    x = x0 + 0.5 * a * t**2
    v = a * t

    def update(i):
        draw_state1(x[i], t[i], v[i])
        return (
            block1,
            push_arrow1,
            friction_arrow1,
            normal_arrow1,
            weight_arrow1,
            info1
        )

    animation1 = FuncAnimation(
        fig1,
        update,
        frames=len(t),
        interval=20,
        blit=False,
        repeat=False
    )

    fig1.canvas.draw_idle()

def reset_animation1(_=None):
    global animation1
    if animation1 is not None and animation1.event_source is not None:
        animation1.event_source.stop()
    draw_state1()

run_button1.on_click(run_animation1)
reset_button1.on_click(reset_animation1)

mass1.observe(lambda change: reset_animation1(), names='value')
push_force1.observe(lambda change: reset_animation1(), names='value')
mu1.observe(lambda change: reset_animation1(), names='value')
show_forces1.observe(lambda change: draw_state1(), names='value')

draw_state1()
plt.show()


## 2. Pulling a block with a rope at an angle

The pulling force is resolved into components:

$$F_x = F\cos\theta$$

$$F_y = F\sin\theta$$

The upward component reduces the normal reaction:

$$N = mg - F_y$$

and therefore

$$F_f = \mu N$$

$$F_{net,x} = F_x - F_f$$

$$a = \frac{F_{net,x}}{m}$$


In [ ]:
# Controls — Animation 2
mass2 = widgets.IntSlider(
    value=20, min=5, max=100, step=5,
    description='Mass (kg):', continuous_update=False,
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

rope_force2 = widgets.FloatSlider(
    value=150, min=0, max=500, step=10,
    description='Rope force (N):', continuous_update=False,
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

angle2 = widgets.IntSlider(
    value=30, min=0, max=70, step=5,
    description='Angle (°):', continuous_update=False,
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

mu2 = widgets.FloatSlider(
    value=0.20, min=0, max=1.0, step=0.05,
    description='Friction μ:', continuous_update=False,
    readout_format='.2f',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='450px')
)

show_forces2 = widgets.Checkbox(value=True, description='Show force arrows')
show_components2 = widgets.Checkbox(value=True, description='Show Fx and Fy components')

run_button2 = widgets.Button(description='Run animation', button_style='primary')
reset_button2 = widgets.Button(description='Reset')

display(widgets.VBox([
    mass2,
    rope_force2,
    angle2,
    mu2,
    show_forces2,
    show_components2,
    widgets.HBox([run_button2, reset_button2])
]))

# Fixed animation frame
X_MIN2, X_MAX2 = 0, 20
Y_MIN2, Y_MAX2 = -3, 9

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.set_xlim(X_MIN2, X_MAX2)
ax2.set_ylim(Y_MIN2, Y_MAX2)
ax2.set_xlabel('Horizontal position (m)')
ax2.set_yticks([])
ax2.set_title('Rope Pulling at an Angle')

ax2.axhline(0, color='black', linewidth=2)

w0, h0 = block_dimensions(mass2.value)
block2 = Rectangle(
    (1, 0), w0, h0,
    facecolor='lightsteelblue',
    edgecolor='black',
    linewidth=1.5
)
ax2.add_patch(block2)

rope_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                              linewidth=2.2, color='tab:blue')
fx_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                            linewidth=2, linestyle='--', color='tab:orange')
fy_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                            linewidth=2, linestyle='--', color='tab:orange')
friction_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                  linewidth=2.2, color='tab:red')
normal_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                linewidth=2.2, color='tab:green')
weight_arrow2 = FancyArrowPatch((0,0),(0,0), arrowstyle='-|>', mutation_scale=18,
                                linewidth=2.2, color='tab:purple')

for arrow in [
    rope_arrow2, fx_arrow2, fy_arrow2,
    friction_arrow2, normal_arrow2, weight_arrow2
]:
    ax2.add_patch(arrow)

info2 = ax2.text(
    0.98, 0.97, '',
    transform=ax2.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    family='monospace',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9)
)

legend_handles2 = [
    Line2D([0],[0], color='tab:blue', lw=2.5, label='Rope force'),
    Line2D([0],[0], color='tab:orange', lw=2, ls='--', label='Fx and Fy'),
    Line2D([0],[0], color='tab:red', lw=2.5, label='Friction'),
    Line2D([0],[0], color='tab:green', lw=2.5, label='Normal reaction'),
    Line2D([0],[0], color='tab:purple', lw=2.5, label='Weight')
]
ax2.legend(handles=legend_handles2, loc='upper left')

animation2 = None

def physics2():
    m = mass2.value
    F = rope_force2.value
    theta = np.deg2rad(angle2.value)
    mu = mu2.value

    Fx = F * np.cos(theta)
    Fy = F * np.sin(theta)

    W = m * g
    N = max(W - Fy, 0)
    friction_max = mu * N

    if Fx <= friction_max:
        friction = Fx
        net = 0
        a = 0
        moving = False
    else:
        friction = friction_max
        net = Fx - friction
        a = net / m
        moving = True

    return m, F, theta, mu, Fx, Fy, W, N, friction, net, a, moving

def draw_state2(x=1, t=0, v=0):
    m, F, theta, mu, Fx, Fy, W, N, friction, net, a, moving = physics2()

    width, height = block_dimensions(m)
    block2.set_x(x)
    block2.set_y(0)
    block2.set_width(width)
    block2.set_height(height)

    cx = x + width / 2
    cy = height / 2
    origin = (x + width, height * 0.72)
    scale = 0.010

    show = show_forces2.value
    components = show and show_components2.value

    rope_end = (
        origin[0] + Fx * scale,
        origin[1] + Fy * scale
    )

    update_arrow(
        rope_arrow2,
        origin,
        rope_end,
        show and F > 0
    )

    update_arrow(
        fx_arrow2,
        origin,
        (origin[0] + Fx * scale, origin[1]),
        components and Fx > 0
    )

    update_arrow(
        fy_arrow2,
        origin,
        (origin[0], origin[1] + Fy * scale),
        components and Fy > 0
    )

    update_arrow(
        friction_arrow2,
        (x, cy),
        (x - friction * scale, cy),
        show and friction > 0
    )

    update_arrow(
        normal_arrow2,
        (cx, height),
        (cx, height + N * scale),
        show and N > 0
    )

    update_arrow(
        weight_arrow2,
        (cx, 0),
        (cx, -W * scale),
        show and W > 0
    )

    state = 'MOVING' if moving else 'AT REST'

    info2.set_text(
        f'm = {m:.0f} kg\n'
        f'Rope force = {F:.1f} N\n'
        f'Angle = {np.rad2deg(theta):.0f}°\n'
        f'Fx = {Fx:.1f} N\n'
        f'Fy = {Fy:.1f} N\n'
        f'N = {N:.1f} N\n'
        f'Friction = {friction:.1f} N\n'
        f'Net Fx = {net:.1f} N\n'
        f'a = {a:.2f} m/s²\n'
        f'v = {v:.2f} m/s\n'
        f't = {t:.2f} s\n'
        f'{state}'
    )

    fig2.canvas.draw_idle()

def run_animation2(_=None):
    global animation2

    m, F, theta, mu, Fx, Fy, W, N, friction, net, a, moving = physics2()

    if not moving or a <= 0:
        draw_state2()
        return

    width, height = block_dimensions(m)
    x0 = 1
    x_end = X_MAX2 - width - 0.5

    t_end = np.sqrt(2 * (x_end - x0) / a)
    t = np.linspace(0, t_end, max(80, int(t_end * 35)))
    x = x0 + 0.5 * a * t**2
    v = a * t

    def update(i):
        draw_state2(x[i], t[i], v[i])
        return (
            block2,
            rope_arrow2,
            fx_arrow2,
            fy_arrow2,
            friction_arrow2,
            normal_arrow2,
            weight_arrow2,
            info2
        )

    animation2 = FuncAnimation(
        fig2,
        update,
        frames=len(t),
        interval=20,
        blit=False,
        repeat=False
    )

    fig2.canvas.draw_idle()

def reset_animation2(_=None):
    global animation2
    if animation2 is not None and animation2.event_source is not None:
        animation2.event_source.stop()
    draw_state2()

run_button2.on_click(run_animation2)
reset_button2.on_click(reset_animation2)

mass2.observe(lambda change: reset_animation2(), names='value')
rope_force2.observe(lambda change: reset_animation2(), names='value')
angle2.observe(lambda change: reset_animation2(), names='value')
mu2.observe(lambda change: reset_animation2(), names='value')
show_forces2.observe(lambda change: draw_state2(), names='value')
show_components2.observe(lambda change: draw_state2(), names='value')

draw_state2()
plt.show()
